# 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import datetime
import inflection
import math

from IPython.display import Image

## 0.1. Helper Functions and Variables

In [ ]:
columns = lambda x: inflection.underscore(x)

## 0.2. Loading Data

In [ ]:
df_sales_raw = pd.read_csv('data/train.csv', low_memory=False)
df_store_raw = pd.read_csv('data/store.csv', low_memory=False)

df = df_sales_raw.merge(df_store_raw, on="Store")
EPS = 1e-6

In [ ]:
df.head()

# 1.0. Data Description

## 1.1. Rename Columns

In [ ]:
new_cols = [columns(col) for col in df.columns]
df.columns = new_cols

In [ ]:
df.columns

## 1.2. Data Dimensions

In [ ]:
df.shape

## 1.3. Data Types

In [ ]:
df.dtypes

In [ ]:
df['date'] = pd.to_datetime(df['date'], format='%Y-%m-%d')

## 1.4. Check NA

In [ ]:
df.isnull().sum()

## 1.5. Fillout NA

In [ ]:
# competition_distance
INF_COMP_DISTANCE = max(df['competition_distance']) / EPS
df.loc[df['competition_distance'].isnull(), 'competition_distance'] = INF_COMP_DISTANCE

# competition_open_since_month
df['competition_open_since_month'] = df.apply(lambda x: x['date'].month if math.isnan(x['competition_open_since_month']) else x['competition_open_since_month'], axis=1)

# competition_open_since_year
df['competition_open_since_year'] = df.apply(lambda x: x['date'].year if math.isnan(x['competition_open_since_year']) else x['competition_open_since_year'], axis=1)

# promo2_since_week
df['promo2_since_week'] = df.apply(lambda x: x['date'].week if math.isnan(x['promo2_since_week']) else x['promo2_since_week'], axis=1)

# promo2_since_year
df['promo2_since_year'] = df.apply(lambda x: x['date'].year if math.isnan(x['promo2_since_year']) else x['promo2_since_year'], axis=1)

# promo_interval
df['promo_interval'] = df['promo_interval'].fillna(0)

month_map = {
    1: 'Jan',
    2: 'Feb',
    3: 'Mar',
    4: 'Apr',
    5: 'May',
    6: 'Jun',
    7: 'Jul',
    8: 'Aug',
    9: 'Sep',
    10: 'Oct',
    11: 'Nov',
    12: 'Dec'
}

df['month_map'] = df['date'].dt.month.map(month_map)

df['is_promo'] = df.apply(lambda x: 1 if ((x['promo_interval'] != 0) and (x['month_map'] in x['promo_interval'])) else 0, axis=1)


In [ ]:
df.sample(5).T

## 1.6. Change Data Types

In [ ]:
df.dtypes

In [ ]:
df["competition_open_since_month"] = df["competition_open_since_month"].astype(int)
df["competition_open_since_year"] = df["competition_open_since_year"].astype(int)
df["promo2_since_week"] = df["promo2_since_week"].astype(int)
df["promo2_since_year"] = df["promo2_since_year"].astype(int)

## 1.7. Descriptive Statistics

In [ ]:
num_attributes = df.select_dtypes(include=['int64', 'float64'])
cat_attributes = df.select_dtypes(exclude=['int64', 'float64', 'datetime64[ns]'])

### 1.7.1. Numeric Attributes

In [ ]:
mean = pd.DataFrame(num_attributes.apply(np.mean)).T
median = pd.DataFrame(num_attributes.apply(np.median)).T
std = pd.DataFrame(num_attributes.apply(np.std)).T
min = pd.DataFrame(num_attributes.apply(min)).T
max = pd.DataFrame(num_attributes.apply(max)).T
range = pd.DataFrame(num_attributes.apply(lambda x: x.max() - x.min())).T
skew = pd.DataFrame(num_attributes.apply(lambda x: x.skew())).T
kurt = pd.DataFrame(num_attributes.apply(lambda x: x.kurt())).T

In [ ]:
df_num_metrics = pd.concat([mean, median, std, min, max, range, skew, kurt]).T
columns = ['mean', 'median', 'std', 'min', 'max', 'range', 'skew', 'kurt']
df_num_metrics.columns = columns

In [ ]:
df_num_metrics

In [ ]:
sns.displot(num_attributes['sales'], kind='kde')

### 1.7.2 Categorical Attributes

In [ ]:
cat_attributes.apply(lambda x: len(x.unique()))

In [ ]:
aux = df[(df['state_holiday'] != '0') & (df['sales'] > 0)]

plt.figure(figsize=(18, 5))

plt.subplot(1, 3, 1)
sns.boxplot(x='state_holiday', y='sales', data=aux)

plt.subplot(1, 3, 2)
sns.boxplot(x='store_type', y='sales', data=aux)

plt.subplot(1, 3, 3)
sns.boxplot(x='assortment', y='sales', data=aux)

# 2. Feature Engineering (early steps)

## 2.1. Hypothesis MindMap

In [ ]:
Image('img/hypothesis-mindmap-eng.png')

## 2.2. Hypothesis

### 2.2.1. Store

**1.** Store with a bigger amount of employees will sell less

**2.** Stores with a bigger stock capacity should sell more

**3.** Stores with a bigger size should sell more

**4.** Stores with a bigger assortment should sell more

**5.** Stores with closer competitors should sell less

**6.** Stores with competitors for a longer time should sell more

### 2.2.1. Product

**1.** Stores that invest more in marketing should sell more.

**2.** Stores with greater product exposure should sell more.

**3.** Stores with lower-priced products should sell more.

**4.** Stores with more aggressive promotions (lower prices) should sell more.

**5.** Stores with longer-running promotions should sell more.

**6.** Stores with more days of promotions should sell more.

**7.** Stores with more consecutive promotions should sell more.

### 2.2.1. Time

**1.** Stores open during the Christmas holidays should sell more.

**2.** Stores should sell more throughout the years.

**3.** Stores should sell more in the second half of the year.

**4.** Stores should sell more after the 10th of each month.

**5.** Stores should sell less on weekends.

**6.** Stores should sell less during school holidays.

## 2.3. Final List of Hypothesis

**1.** Stores with a wider assortment should sell more.

**2.** Stores with closer competitors should sell less.

**3.** Stores with competitors who have been around longer should sell more.

**4.** Stores with promotions running for longer periods should sell more.

**5.** Stores with more days of promotions should sell more.

**6.** Stores with more consecutive promotions should sell more.

**7.** Stores open during the Christmas holidays should sell more.

**8.** Stores should sell more throughout the year.

**9.** Stores should sell more in the second half of the year.

**10.** Stores should sell more after the 10th of each month.

**11.** Stores should sell less on weekends.

**12.** Stores should sell less during school holidays.

In [ ]:
# year
# month
# day
# week_of_year
# year_week

# competition since
# competition time month
# promo since

# assortment
# state holiday